# Object detection (IoU (Intersection over Union), YOLO (You Only Look Once))

_Deep Learning_

**Not just 'is there a cat?' but 'where is it?' — draw a box and score how well it fits.**

Classification asks "what is in the image?". Object detection also asks "where?", drawing a bounding box around each object.

     To score a predicted box against the true box, we use Intersection-over-Union (IoU): how much they overlap.

---

This notebook builds the IoU overlap score up one step at a time. Run each cell, read the note above it, and watch how a single ratio captures how well a predicted box fits the true one. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — NumPy

Intersection-over-Union scores how well two boxes overlap: the area they share divided by the area they jointly cover. We build it in two steps: (1) write the IoU function, computing the intersection and union, then (2) score a few example boxes against a ground-truth box.

### Step 1 — Write the IoU function

Each box is `(x1, y1, x2, y2)` — top-left and bottom-right corners. The overlap rectangle's left/top edge is the *larger* of the two left/top edges, and its right/bottom edge is the *smaller* of the two right/bottom edges. Overlap width and height are clamped at 0 with `max(0, ...)` so non-touching boxes score zero. The **union** is the two areas minus the shared intersection (so overlap isn't double-counted), and IoU is intersection over union.

In [ ]:
import numpy as np

def iou(a, b):
    # boxes as (x1, y1, x2, y2)
    ix1 = max(a[0], b[0])   # left edge of overlap = rightmost of the two lefts
    iy1 = max(a[1], b[1])   # top edge of overlap
    ix2 = min(a[2], b[2])   # right edge of overlap = leftmost of the two rights
    iy2 = min(a[3], b[3])   # bottom edge of overlap

    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)   # overlap area (0 if no overlap)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter                 # combined area, overlap counted once
    return inter / union

### Step 2 — Score a few example boxes

We score three predictions against one ground-truth box. An identical box gives IoU `1.0`; a slightly shifted box gives a high but imperfect score; a box that never touches gives `0.0`. This is the single number detectors use to decide whether a prediction counts as a hit.

In [ ]:
truth = (10, 10, 50, 50)        # ground-truth box

print("perfect :", iou(truth, (10, 10, 50, 50)))        # 1.0
print("good    :", round(iou(truth, (15, 15, 55, 55)), 3))
print("no overlap:", iou(truth, (100, 100, 140, 140)))  # 0.0

## Visualize the data & results

_How well does each predicted box overlap the real bounding box of a handwritten digit?_

### Step 1 — Derive a real bounding box from a digit image

We take one real 8x8 image of a handwritten 0 and find every pixel brighter than 2. The smallest and largest x/y of those bright pixels give a tight `(x1, y1, x2, y2)` bounding box around the ink — our ground-truth box.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.images[0]                   # real 8x8 image of a 0

ys, xs = np.nonzero(img > 2)             # coordinates of bright pixels
truth = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))

### Step 2 — Define IoU and build a range of predictions

We redefine the same IoU function compactly, then create five predicted boxes relative to the truth: a perfect copy, a slightly tight box, a slightly loose box, a shifted box, and one with no overlap at all. This spread lets us see how IoU responds as a prediction drifts from the target.

In [ ]:
def iou(a, b):                           # boxes as (x1, y1, x2, y2)
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua

preds = {"perfect": truth,
         "tight": tuple(t + 1 for t in truth),
         "loose": (truth[0]-1, truth[1]-1, truth[2]+2, truth[3]+2),
         "shifted": tuple(t + 3 for t in truth),
         "no overlap": (truth[2]+2, truth[3]+2, truth[2]+5, truth[3]+5)}

### Step 3 — Plot the IoU of each prediction

We score every predicted box against the truth and chart the results. The bar heights fall off as the boxes drift — perfect and tight stay near 1, loose and shifted drop, and the non-overlapping box bottoms out at 0 — making concrete how IoU rewards a good fit and punishes a bad one.

In [ ]:
labels = list(preds)
values = [iou(truth, b) for b in preds.values()]
colors = ["#7ee787", "#7ee787", "#ffb454", "#ff7b72", "#ff7b72"]

plt.bar(labels, values, color=colors)
plt.title("IoU of predicted boxes vs the real digit's bounding box")
plt.show()